In [ ]:
import json
import re
import httpx

async def test_endpoints():
    test_isbns = ["9783161484100", "9783730600405", "3446144900"]
    bibkeys_str = ",".join([f"ISBN:{isbn}" for isbn in test_isbns])

    async with httpx.AsyncClient(follow_redirects=True) as client:
        
        # ==========================================================
        # Test 1: Google Books View API (JSONP / Batch Query)
        # ==========================================================
        print("--- TESTING GOOGLE BOOKS VIEW API (JSONP) ---")
        google_url = f"https://books.google.com/books?bibkeys={bibkeys_str}&jscmd=viewapi"
        try:
            res = await client.get(google_url, timeout=5.0)
            print(f"Google Books Status: {res.status_code}")
            if res.status_code == 200:
                raw_text = res.text.strip()
                # JSONP usually wraps data like: _GBSBookInfo({...}); or similar callback
                # Let's use regex to extract the inner JSON object
                match = re.search(r'^[a-zA-Z0-9_]+\s*\(\s*(.*)\s*\)\s*;?$', raw_text, re.DOTALL)
                if match:
                    json_str = match.group(1)
                    parsed_data = json.loads(json_str)
                    print("Successfully parsed JSONP from Google Books!")
                    for key, val in parsed_data.items():
                        thumb = val.get("thumbnail_url", "No thumbnail")
                        print(f" -> {key}: {thumb}")
                else:
                    print("Could not match JSONP wrapper pattern. Raw response snippet:", raw_text[:100])
        except Exception as e:
            print(f"Google Books Error: {e}")

        print("\n--- TESTING OPEN LIBRARY BOOKS API (format=json) ---")
        # ==========================================================
        # Test 2: Open Library Books API (Batch clean JSON query)
        # ==========================================================
        ol_books_url = f"https://openlibrary.org/api/books?bibkeys={bibkeys_str}&format=json&jscmd=viewapi"
        try:
            res = await client.get(ol_books_url, timeout=5.0)
            print(f"Open Library Books API Status: {res.status_code}")
            if res.status_code == 200:
                ol_data = res.json()
                print("Successfully fetched Open Library batch JSON!")
                for key, val in ol_data.items():
                    thumb = val.get("thumbnail_url", "No thumbnail")
                    print(f" -> {key}: {thumb}")
        except Exception as e:
            print(f"Open Library Books API Error: {e}")

await test_endpoints()

--- TESTING GOOGLE BOOKS VIEW API (JSONP) ---
Google Books Status: 200
Could not match JSONP wrapper pattern. Raw response snippet: var _GBSBookInfo = {"ISBN:9783161484100":{"bib_key":"ISBN:9783161484100","info_url":"https://books.g

--- TESTING OPEN LIBRARY BOOKS API (format=json) ---
Open Library Books API Status: 200
Successfully fetched Open Library batch JSON!
 -> ISBN:9783161484100: https://covers.openlibrary.org/b/id/15179814-S.jpg
 -> ISBN:9783730600405: No thumbnail
 -> ISBN:3446144900: https://covers.openlibrary.org/b/id/2205085-S.jpg


In [ ]:
async def test_waterfall_providers():
    TEST_ISBNS = [
        "3446144900",
        "9783209115393"
        #"9783161484100",
        #"9783730600405"
    ]
    
    async with httpx.AsyncClient(follow_redirects=True) as client:
        for isbn in TEST_ISBNS:
            print(f"\n==============================")
            print(f"TESTING ISBN: {isbn}")
            print(f"==============================")
            hit_found = False

            # 1. Test DNB
            dnb_url = f"https://portal.dnb.de/opac/mvb/cover?isbn={isbn}&size=m"
            try:
                res = await client.head(dnb_url, timeout=3.0)
                print(f"[DNB] Status: {res.status_code} | Content-Type: {res.headers.get('content-type')}")
                if res.status_code == 200 and "image" in res.headers.get("content-type", ""):
                    print(f" -> SUCCESS via DNB: {res.url}")
                    hit_found = True
            except Exception as e:
                print(f"[DNB] Error: {e}")

            if hit_found:
                continue

            # 2. Test Open Library
            ol_url = f"https://covers.openlibrary.org/b/isbn/{isbn}-M.jpg?default=false"
            try:
                res = await client.head(ol_url, timeout=3.0)
                print(f"[Open Library] Status: {res.status_code} | Content-Type: {res.headers.get('content-type')}")
                if res.status_code == 200 and "image" in res.headers.get("content-type", ""):
                    print(f" -> SUCCESS via Open Library: {res.url}")
                    hit_found = True
            except Exception as e:
                print(f"[Open Library] Error: {e}")

            if hit_found:
                continue

            # 3. Test Google Books Volumes API
            google_url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
            try:
                res = await client.get(google_url, timeout=3.0)
                print(f"[Google Books] Status: {res.status_code}")
                if res.status_code == 200:
                    data = res.json()
                    items = data.get("items", [])
                    if items:
                        image_links = items[0].get("volumeInfo", {}).get("imageLinks", {})
                        thumb = image_links.get("thumbnail") or image_links.get("smallThumbnail")
                        if thumb:
                            print(f" -> SUCCESS via Google Books: {thumb}")
                            hit_found = True
                        else:
                            print("[Google Books] Item found, but no image links available.")
                    else:
                        print("[Google Books] No items returned for this ISBN.")
            except Exception as e:
                print(f"[Google Books] Error: {e}")

            if not hit_found:
                print(f" -> RESULT: Cover not found across any provider.")

await test_waterfall_providers()


TESTING ISBN: 3446144900
[DNB] Status: 400 | Content-Type: None
[Open Library] Status: 200 | Content-Type: image/jpeg
 -> SUCCESS via Open Library: https://covers.openlibrary.org/b/isbn/3446144900-M.jpg?default=false

TESTING ISBN: 9783209115393
[DNB] Status: 200 | Content-Type: image/jpeg; charset=UTF-8
 -> SUCCESS via DNB: https://portal.dnb.de/opac/mvb/cover?isbn=9783209115393&size=m


In [2]:
import asyncio
import httpx

async def test_providers():
    TEST_ISBNS = [
        "9783161484100",  # Example academic text
        "9783730600405"   # Der Mann ohne Eigenschaften
    ]
    
    async with httpx.AsyncClient(follow_redirects=True) as client:
        for isbn in TEST_ISBNS:
            print(f"\n==============================")
            print(f"TESTING ISBN: {isbn}")
            print(f"==============================")

            # 1. Test DNB
            dnb_url = f"https://portal.dnb.de/opac/mvb/cover?isbn={isbn}&size=m"
            try:
                res = await client.head(dnb_url, timeout=5.0)
                print(f"[DNB] Status: {res.status_code} | Content-Type: {res.headers.get('content-type')} | Final URL: {res.url}")
            except Exception as e:
                print(f"[DNB] Error: {e}")

            # 2. Test Google Books View API
            google_url = f"https://books.google.com/books?bibkeys=ISBN:{isbn}&jscmd=viewapi&format=json"
            try:
                res = await client.get(google_url, timeout=5.0)
                print(f"[Google Books] Status: {res.status_code}")
                if res.status_code == 200:
                    data = res.json()
                    key = f"ISBN:{isbn}"
                    if key in data and "thumbnail_url" in data[key]:
                        print(f"[Google Books] Found Thumbnail: {data[key]['thumbnail_url']}")
                    else:
                        print(f"[Google Books] JSON response did not contain thumbnail for key {key}.")
            except Exception as e:
                print(f"[Google Books] Error: {e}")

            # 3. Test Open Library
            ol_url = f"https://covers.openlibrary.org/b/isbn/{isbn}-M.jpg?default=false"
            try:
                res = await client.head(ol_url, timeout=5.0)
                print(f"[Open Library] Status: {res.status_code} | Content-Type: {res.headers.get('content-type')} | Final URL: {res.url}")
            except Exception as e:
                print(f"[Open Library] Error: {e}")

# Run directly with await in Jupyter
await test_providers()


TESTING ISBN: 9783161484100
[DNB] Status: 404 | Content-Type: text/html;charset=utf-8 | Final URL: https://portal.dnb.de/opac/mvb/cover?isbn=9783161484100&size=m
[Google Books] Status: 200
[Google Books] Error: Expecting value: line 1 column 1 (char 0)
[Open Library] Status: 200 | Content-Type: image/jpeg | Final URL: https://ia801601.us.archive.org/view_archive.php?archive=/25/items/m_covers_0014/m_covers_0014_31.zip&file=0014317533-M.jpg

TESTING ISBN: 9783730600405
[DNB] Status: 200 | Content-Type: image/jpeg; charset=UTF-8 | Final URL: https://portal.dnb.de/opac/mvb/cover?isbn=9783730600405&size=m
[Google Books] Status: 200
[Google Books] Error: Expecting value: line 1 column 1 (char 0)
[Open Library] Status: 404 | Content-Type: None | Final URL: https://covers.openlibrary.org/b/isbn/9783730600405-M.jpg?default=false
